In [1]:
import json
from openai import OpenAI
import sys
import pandas as pd
import numpy as np
import random
import re
from tqdm import tqdm
import unicodedata
import matplotlib.pyplot as plt
import spacy
from scipy.stats import pearsonr, spearmanr
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.metrics import cohen_kappa_score
from tenacity import retry, stop_after_attempt, wait_exponential
from scipy.stats import pointbiserialr
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from scipy.optimize import milp, LinearConstraint, Bounds
from empath import Empath
from adjustText import adjust_text
import plotly.express as px
import emoji

from nrclex import NRCLex

import anthropic

from numpy import sqrt, std, mean
from scipy.stats import norm

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
SEED = 67

nlp = spacy.load("en_core_web_trf")
empath_lex = Empath()
PSYCH_VOCAB = set()

_nrc = NRCLex()
try:
    nrc_dict = _nrc.lexicon
except AttributeError:
    nrc_dict = _nrc.__lexicon__

In [3]:
#posts_text = pd.read_csv('POSTS_subset_text.csv', encoding='ISO-8859-1')

fullset_essays_text = pd.read_csv('ESSAYS_fullset_text.csv', encoding='ISO-8859-1')
subset_essays_text = pd.read_csv('ESSAYS_subset_text.csv', encoding='ISO-8859-1')

subset_facebook_text = pd.read_csv('FACEBOOK_subset_text.csv', encoding='ISO-8859-1')

In [4]:
len(fullset_essays_text)

2468

In [ ]:
# Implements a limited form of the lexical evidence-channel ablation described
# in the Content-Masking Ablation section, for generating the graph and stats below only.

#commented out sections only apply to the unavailable student introduction post data
def _normalize(text):
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[–—]', '-', text)
    text = re.sub(r"[\u2018\u2019\u02BC]", "'", text)
    text = re.sub(r'[\u201C\u201D]', '"', text)
    return text

LINE_START = r"^[\s\-\*•]*(?:\d+[\.\)]\s*)?"

'''
ALL_TEMPLATE_PATTERNS = [
    re.compile(
        LINE_START +
        r"what do you do when you'?re not in the [REDACTED]\??\s*"
        r"(?:\([^)]*\))?",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"what(?:'?s| is) (?:something|one thing) interesting about you\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"why are you taking "
        r"([REDACTED])?"
        r"([REDACTED]|[REDACTED]|this course)\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START +
        r"what do you hope to (?:get|gain|learn) (?:out )?(?:of|from) this course\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what(?:'?s| is) your name\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"where do you live\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what other [REDACTED] courses have you taken(?:\s+so far)?\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what courses do you plan to take(?:\s+next semester)?\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
        LINE_START + r"what specialization are you planning\??",
        re.IGNORECASE | re.MULTILINE
    ),
    re.compile(
    r"#\s*conn?e+c?t+[\s_\-]*me\b",
    re.IGNORECASE
    ),
]
'''

REPLACEMENT = '_'

'''
def remove_template_questions(text):
    text = _normalize(text)
    for pattern in ALL_TEMPLATE_PATTERNS:
        text = pattern.sub(REPLACEMENT, text)
    text = re.sub(r'(\s*_\s*)+', ' _ ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
'''

PII_PLACEHOLDERS = re.compile(r'\[(?:NAME|EMAIL|PHONE|URL|HANDLE|LOCATION|ORG|DATE|COUNTRY)\]')

def strip_pii_placeholders(text, replacement='_'):
    return PII_PLACEHOLDERS.sub(replacement, text)

EMPATH_CATEGORIES = [

    'positive_emotion', 'joy', 'cheerfulness', 'contentment', 'affection', 'love',
    'optimism', 'pride', 'zest',
    'negative_emotion', 'sadness', 'disappointment', 'suffering', 'torment',
    'nervousness', 'fear', 'timidity',
    'anger', 'rage', 'aggression', 'irritability', 'exasperation', 'hate',
    'disgust', 'shame',
    'sympathy', 'emotional', 'warmth',
    'swearing_terms',

    'thinking', 'order', 'confusion', 'anticipation', 'deception',
]

empath_lex = Empath()
PSYCH_VOCAB = set()
for cat in EMPATH_CATEGORIES:
    PSYCH_VOCAB.update(w.lower() for w in empath_lex.cats.get(cat, []))

NRC_CATEGORIES_KEEP = {'joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise',
                       'positive', 'negative'}

for word, emotions in nrc_dict.items():
    if any(e in NRC_CATEGORIES_KEEP for e in emotions):
        PSYCH_VOCAB.add(word.lower())

print(f"PSYCH_VOCAB size: {len(PSYCH_VOCAB)}")

#PROGRAM_TERMS = [REDACTED]

#COURSE_NUM_PATTERN = re.compile(r'\b[A-Z]{2,4}\d{4}\b')

URL_PATTERNS = [
    re.compile(r'https?://[^\s<>"\'\[\]]+', re.IGNORECASE),
    re.compile(r'www\.[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(?:/[^\s<>"\'\[\]]*)?', re.IGNORECASE),
]

TEXT_EMOTICON_PATTERN = re.compile(
    r"""(?<![a-zA-Z])(?:
        [;:=][-']?[)(DPpOo3/\\|*]    |  # standard :) :D ;P etc.
        [)(DPp]['-]?[;:=]             |  # reversed (: D:
        <3                             |  # heart
        [=][\)(\\/]                    |  # =) =( =/
        >\.<                              # >.<
    )(?![a-zA-Z])""",
    re.VERBOSE
)

FUNCTION_POS = {
    'ADP',
    'AUX',
    'CCONJ',
    'DET',
    'PART',
    'PRON',
    'SCONJ',
    'INTJ',
    'PUNCT',
    'SPACE',
}

FUNCTION_WORD_WHITELIST = {
    'something', 'everything', 'nothing', 'anything',
    'someone', 'everyone', 'no one', 'anyone',
    'somebody', 'everybody', 'nobody', 'anybody',

    'lots', 'much', 'many', 'few', 'several', 'some', 'any',
    'more', 'most', 'less', 'least', 'enough',
    'very', 'really', 'quite', 'pretty', 'too', 'so',

    'not', "n't", 'never', 'no', 'neither', 'nor',

    'also', 'just', 'still', 'already', 'yet', 'even',
    'always', 'often', 'sometimes', 'usually',
    'here', 'there', 'where', 'when', 'how', 'why', 'what', 'who',
    'again', 'then', 'now', 'only', 'else',
    'between', 'beyond', 'ago',
}

REPLACEMENT = '_'


def extract_emoticons(text):
    positions = []

    for m in TEXT_EMOTICON_PATTERN.finditer(text):
        positions.append((m.start(), m.end(), m.group()))

    for m in emoji.emoji_list(text):
        positions.append((m['match_start'], m['match_end'], m['emoji']))

    return sorted(positions, key=lambda x: x[0])


def strip_urls(text):
    for pat in URL_PATTERNS:
        text = pat.sub('', text)
    return text

'''
def replace_program_terms(text):
    text = COURSE_NUM_PATTERN.sub(REPLACEMENT, text)

    for term in sorted(PROGRAM_TERMS, key=len, reverse=True):
        text = re.sub(r'\b' + re.escape(term) + r'\b', REPLACEMENT, text)

    return text
'''

def filter_stats(text):
    text = strip_urls(text)
    #text = remove_template_questions(text)
    text = strip_pii_placeholders(text)
    #text = replace_program_terms(text)

    emoticon_positions = extract_emoticons(text)
    emoticon_ranges = set()
    for start, end, _ in emoticon_positions:
        emoticon_ranges.update(range(start, end))

    doc = nlp(text)
    c = dict(word_tokens=0, kept_function=0, kept_psych=0,
             replaced_num=0, replaced_content=0, emoticons=0)

    for tok in doc:
        tok_chars = set(range(tok.idx, tok.idx + len(tok.text)))
        if tok_chars & emoticon_ranges:
            c['emoticons'] += 1
            continue
        if tok.pos_ in ('SPACE', 'PUNCT') or tok.is_space or tok.is_punct:
            continue

        c['word_tokens'] += 1
        if tok.pos_ in FUNCTION_POS or tok.text.lower() in FUNCTION_WORD_WHITELIST:
            c['kept_function'] += 1
        elif tok.lemma_.lower() in PSYCH_VOCAB or tok.text.lower() in PSYCH_VOCAB:
            c['kept_psych'] += 1
        elif tok.pos_ == 'NUM':
            c['replaced_num'] += 1
        else:
            c['replaced_content'] += 1
    return c


def preservation_table(corpora):
    rows = []
    for name, texts in corpora.items():
        stats = pd.DataFrame(filter_stats(t) for t in texts)
        wt = stats['word_tokens'].replace(0, np.nan)
        rows.append({
            'corpus': name,
            'n_docs': len(stats),
            'mean_word_tokens': stats['word_tokens'].mean(),
            'preservation_rate': ((stats['kept_function'] + stats['kept_psych']) / wt).mean(),
            'psych_rate': (stats['kept_psych'] / wt).mean(),
            'mean_psych_tokens': stats['kept_psych'].mean(),
            'pct_docs_zero_psych': (stats['kept_psych'] == 0).mean() * 100,
            'mean_emoticons': stats['emoticons'].mean(),
        })
    return pd.DataFrame(rows)

'''
corpora = {
    'Posts': (
        POSTS_subset_text
        .iloc[:, -1]
        .dropna()
        .astype(str)
        .tolist()
    ),
    'Essays': (
        fullset_essays_text
        .iloc[:, -1]
        .dropna()
        .astype(str)
        .tolist()
    ),
    'Facebook': (
        subset_facebook_text['STATUS']
        .dropna()
        .astype(str)
        .tolist()
    ),
}
'''

corpora = {
    'Essays': (
        fullset_essays_text
        .iloc[:, -1]
        .dropna()
        .astype(str)
        .tolist()
    ),
    'Facebook': (
        subset_facebook_text['STATUS']
        .dropna()
        .astype(str)
        .tolist()
    ),
}

print(preservation_table(corpora).round(3))

def before_after_means(corpora):
    rows = []
    for name, texts in corpora.items():
        s = pd.DataFrame(filter_stats(t) for t in texts)
        before = s['word_tokens']
        after  = s['kept_function'] + s['kept_psych']
        rows.append({
            'corpus': name,
            'n_docs': len(s),
            'mean_before': before.mean(),
            'mean_after':  after.mean(),
            'mean_survival_rate': (after / before.replace(0, np.nan)).mean(),
        })
    return pd.DataFrame(rows)

print(before_after_means(corpora).round(3))

<>:15: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<>:70: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:15: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
<>:70: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_14905/2780703355.py:15: SyntaxWarning: "\?" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\?"? A raw string is also an option.
  r"what do you do when you'?re not in the [REDACTED]\??\s*"
/tmp/ipykernel_14905/2780703355.py:70: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did

PSYCH_VOCAB size: 6559


In [ ]:
def get_empath_counts_and_rates(
        df,
        categories,
        corpus_name
    ):
    text_df = df.loc[
        :, ~df.columns.astype(str).str.startswith("Unnamed:")
    ]

    if text_df.shape[1] != 1:
        raise ValueError(
            f"{corpus_name}: expected one text column after removing "
            f"CSV index columns, found {text_df.columns.tolist()}"
        )
    
    text_series = (
        text_df.iloc[:, 0]
        .dropna()
        .astype(str)
        .reset_index(drop=True)
    )

    counts = text_series.apply(
        lambda text: empath_lex.analyze(
            text,
            categories=categories,
            normalize=False
        )
    )

    counts_df = pd.DataFrame(counts.tolist()).fillna(0)

    word_counts = (
        text_series
        .str.findall(r"\b\w+\b")
        .str.len()
        .replace(0, np.nan)
    )

    rates_df = counts_df.div(word_counts, axis=0) * 100

    return {
        "corpus": corpus_name,
        "texts": text_series,
        "counts_df": counts_df,
        "word_counts": word_counts,
        "rates_df": rates_df
    }


def category_diagnostics(results):
    counts_df = results["counts_df"]
    rates_df = results["rates_df"]
    word_counts = results["word_counts"]

    n_texts = len(counts_df)
    total_words = word_counts.sum()

    diagnostics = pd.DataFrame({
        "corpus": results["corpus"],
        "category": counts_df.columns,
        "total_hits": counts_df.sum(axis=0).values,
        "total_words": total_words,
        "rate_per_100_words": (counts_df.sum(axis=0) / total_words * 100).values,
        "docs_with_hit": (counts_df > 0).sum(axis=0).values,
        "doc_prevalence_pct": ((counts_df > 0).sum(axis=0) / n_texts * 100).values,
        "median_doc_rate": rates_df.median(axis=0).values,
        "mean_doc_rate": rates_df.mean(axis=0).values,
        "max_doc_rate": rates_df.max(axis=0).values
    })

    return diagnostics.sort_values(
        ["doc_prevalence_pct", "total_hits"],
        ascending=[True, True]
    )

LIWC_LIKE_CATEGORIES = [
    'positive_emotion', 'joy', 'cheerfulness', 'contentment', 'affection', 'love',
    'optimism', 'pride', 'zest',
    'negative_emotion', 'sadness', 'disappointment', 'suffering', 'torment',
    'nervousness', 'fear', 'timidity',
    'anger', 'rage', 'aggression', 'irritability', 'exasperation', 'hate',
    'disgust', 'shame',
    'sympathy', 'emotional', 'warmth',
    'swearing_terms',

    'thinking', 'order', 'confusion', 'anticipation', 'deception',
]

essay_results = get_empath_counts_and_rates(
    fullset_essays_text,
    categories=LIWC_LIKE_CATEGORIES,
    corpus_name="Essays"
)
'''
post_results = get_empath_counts_and_rates(
    posts_text,
    categories=LIWC_LIKE_CATEGORIES,
    corpus_name="Posts"
)
'''
fb_results = get_empath_counts_and_rates(
    subset_facebook_text[['STATUS']],
    categories=LIWC_LIKE_CATEGORIES,
    corpus_name="Facebook"
)

essay_diag = category_diagnostics(essay_results)
#post_diag = category_diagnostics(post_results)
fb_diag = category_diagnostics(fb_results)

'''
diag_all = pd.concat(
    [essay_diag, post_diag, fb_diag],
    ignore_index=True
)
'''
diag_all = pd.concat(
    [essay_diag, fb_diag],
    ignore_index=True
)

print(diag_all.head())

In [ ]:
'''
diag_all = pd.concat(
    [essay_diag, post_diag, fb_diag],
    ignore_index=True
)
'''

diag_all = pd.concat(
    [essay_diag, fb_diag],
    ignore_index=True
)


marker_map = {
    "Essays": "o",
    "Posts": "s",
    "Facebook": "^",
}

color_map = {
    "Essays": "#1f77b4",
    "Posts": "#2ca02c",
    "Facebook": "#ff7f0e",
}

fig, ax = plt.subplots(figsize=(12, 5))

texts = []

for corpus_name in ["Essays", "Posts", "Facebook"]:
    group = diag_all[diag_all["corpus"] == corpus_name]

    if group.empty:
        continue

    ax.scatter(
        group["doc_prevalence_pct"],
        group["rate_per_100_words"],
        label=corpus_name,
        marker=marker_map[corpus_name],
        color=color_map[corpus_name],
        s=25,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.25,
        zorder=3,
    )

for _, row in labels_to_show.iterrows():
    texts.append(
        ax.text(
            row["doc_prevalence_pct"],
            row["rate_per_100_words"],
            row["category"],
            fontsize=8,
            color="#222222",
            zorder=4,
        )
    )

adjust_text(
    texts,
    ax=ax,
    arrowprops={
        "arrowstyle": "-",
        "lw": 0.4,
        "color": "#666666",
        "alpha": 0.6,
    },
)

ax.set_xlabel("Documents containing the category (%)")
ax.set_ylabel("Category occurrences per 100 words")

ax.set_xlim(-5, 102)
ax.set_xticks(np.arange(0, 101, 20))

ax.grid(
    True,
    color="#cccccc",
    linewidth=0.5,
    alpha=0.2,
    zorder=0,
)

ax.legend(
    title="Corpus",
    loc="upper right",
    frameon=True,
    fontsize=8,
    title_fontsize=9,
)

fig.tight_layout()

fig.savefig(
    "category_prevalence_density.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.03,
)

plt.show()